# Resultados y Robustez

Metricas, visualizaciones y test de Monte Carlo del algoritmo (vectorizado, sin bucles for).


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import time

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 50)


In [ ]:
# --- Cargar resultados de la estrategia (desde ejecucionCostes) ---
hist = pd.read_csv('historial_cartera.csv')
hist['date'] = pd.to_datetime(hist['date'])
hist = hist.sort_values('date').set_index('date')

strategy_ret = hist['portfolio_value'].pct_change().dropna()
strategy_cum = (1 + strategy_ret).cumprod()

strategy_ret.head()


In [ ]:
# --- Benchmark SPY ---
spy = yf.download('SPY', start='2015-01-01', auto_adjust=True, progress=False)
if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.get_level_values(0)
spy = spy.reset_index().rename(columns={'Date':'date'})
spy['date'] = pd.to_datetime(spy['date'])
spy = spy.set_index('date')
spy_m = spy.resample('ME').last()
spy_ret = spy_m['Close'].pct_change().dropna()

# Alinear fechas
common_idx = strategy_ret.index.intersection(spy_ret.index)
strategy_ret = strategy_ret.loc[common_idx]
spy_ret = spy_ret.loc[common_idx]
strategy_cum = (1 + strategy_ret).cumprod()
spy_cum = (1 + spy_ret).cumprod()

print('Fechas comunes:', common_idx.min(), '->', common_idx.max(), 'meses:', len(common_idx))


In [ ]:
# --- Metricas ---
def cagr(returns, periods_per_year=12):
    n = len(returns)
    if n == 0:
        return np.nan
    total = (1 + returns).prod()
    years = n / periods_per_year
    return total ** (1 / years) - 1

def vol(returns, periods_per_year=12):
    return returns.std() * np.sqrt(periods_per_year)

def sharpe(returns, rf=0.0, periods_per_year=12):
    excess = returns - rf / periods_per_year
    return np.sqrt(periods_per_year) * excess.mean() / excess.std()

def sortino(returns, rf=0.0, periods_per_year=12):
    excess = returns - rf / periods_per_year
    downside = excess[excess < 0]
    downside_std = downside.std()
    return np.sqrt(periods_per_year) * excess.mean() / downside_std

def max_drawdown(cum_returns):
    peak = cum_returns.cummax()
    dd = (cum_returns / peak) - 1
    return dd.min()

cov = np.cov(strategy_ret, spy_ret, ddof=0)[0, 1]
var = np.var(spy_ret, ddof=0)
beta = cov / var if var > 0 else np.nan
alpha = (strategy_ret.mean() - beta * spy_ret.mean()) * 12

metrics = pd.DataFrame({
    'CAGR': [cagr(strategy_ret), cagr(spy_ret)],
    'Volatilidad': [vol(strategy_ret), vol(spy_ret)],
    'Sharpe': [sharpe(strategy_ret), sharpe(spy_ret)],
    'Sortino': [sortino(strategy_ret), sortino(spy_ret)],
    'Max Drawdown': [max_drawdown(strategy_cum), max_drawdown(spy_cum)],
    'Beta': [beta, 1.0],
    'Alpha': [alpha, 0.0],
}, index=['Estrategia', 'SPY'])

metrics


In [ ]:
# --- Visualizaciones ---
plt.figure(figsize=(10, 5))
(strategy_cum - 1).mul(100).plot(label='Estrategia')
(spy_cum - 1).mul(100).plot(label='SPY')
plt.title('Rentabilidad acumulada (%)')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(strategy_ret, bins=30, alpha=0.6, label='Estrategia')
plt.hist(spy_ret, bins=30, alpha=0.6, label='SPY')
plt.title('Histograma de retornos mensuales')
plt.legend()
plt.grid(True)
plt.show()

strategy_year = (1 + strategy_ret).resample('YE').prod() - 1
spy_year = (1 + spy_ret).resample('YE').prod() - 1

plt.figure(figsize=(6, 6))
plt.scatter(spy_year, strategy_year, alpha=0.7)
plt.axhline(0, color='grey', linewidth=1)
plt.axvline(0, color='grey', linewidth=1)
plt.title('Retornos anuales: Estrategia vs SPY')
plt.xlabel('SPY')
plt.ylabel('Estrategia')
plt.grid(True)
plt.show()

strategy_quarter = (1 + strategy_ret).resample('QE').prod() - 1
spy_quarter = (1 + spy_ret).resample('QE').prod() - 1

plt.figure(figsize=(6, 6))
plt.scatter(spy_quarter, strategy_quarter, alpha=0.7)
plt.axhline(0, color='grey', linewidth=1)
plt.axvline(0, color='grey', linewidth=1)
plt.title('Retornos trimestrales: Estrategia vs SPY')
plt.xlabel('SPY')
plt.ylabel('Estrategia')
plt.grid(True)
plt.show()


## Test de Monte Carlo (>= 25M monos, vectorizado)

- Coste de rebalanceo: 0.23% x2 mensual (0.46%) aplicado al 100% del capital.
- Sin minimo de 23 USD por orden.


In [ ]:
# Matriz de retornos log mensuales para 9 sectores + GLD
df = pd.read_pickle('sp500_history_filtered.pkl')
df['date'] = pd.to_datetime(df['date'])

monthly = (
    df.set_index('date')
      .groupby('symbol')
      .resample('ME')
      .last()
      .reset_index()
)
monthly['log_ret'] = np.log(monthly['close'] / monthly.groupby('symbol')['close'].shift(1))
monthly = monthly.dropna(subset=['log_ret'])

SECTORS_9 = [
    'Consumer Discretionary',
    'Consumer Staples',
    'Energy',
    'Financials',
    'Health Care',
    'Industrials',
    'Information Technology',
    'Materials',
    'Utilities',
]
sector_monthly = (
    monthly[monthly['sector'].isin(SECTORS_9)]
        .groupby(['sector', 'date'], as_index=False)['log_ret']
        .mean()
)

gld_raw = yf.download('GLD', start='2015-01-01', auto_adjust=True, progress=False)
if isinstance(gld_raw.columns, pd.MultiIndex):
    gld_raw.columns = gld_raw.columns.get_level_values(0)
gld_raw = gld_raw.reset_index()
gld_df = gld_raw[['Date', 'Close']].rename(columns={'Date':'date','Close':'close'})
gld_df['date'] = pd.to_datetime(gld_df['date'])
gld_df = gld_df.sort_values('date')
gld_monthly = (
    gld_df.set_index('date')
          .resample('ME')
          .last()
          .reset_index()
)
gld_monthly['log_ret'] = np.log(gld_monthly['close'] / gld_monthly['close'].shift(1))
gld_monthly = gld_monthly.dropna(subset=['log_ret'])
gld_monthly['sector'] = 'GLD'

universe = pd.concat([sector_monthly, gld_monthly[['sector','date','log_ret']]], ignore_index=True)
valid_dates = (universe.groupby('date')['sector'].nunique() == 10)
valid_dates = valid_dates[valid_dates].index
universe = universe[universe['date'].isin(valid_dates)]

ret_matrix = (
    universe.pivot(index='date', columns='sector', values='log_ret')
          .sort_index()
)
ret_matrix = ret_matrix[SECTORS_9 + ['GLD']]
ret_matrix.head()


In [ ]:
# Estimacion de memoria 
def estimar_memoria(n_monos, n_assets, n_months):
    BYTES_PER_FLOAT = 4  # float32
    GB = 1024**3
    mem_weights = (n_monos * n_assets * BYTES_PER_FLOAT) / GB
    mem_returns = (n_months * n_monos * BYTES_PER_FLOAT) / GB
    total = mem_weights + mem_returns
    print(f'Memoria pesos (GB): {mem_weights:.2f}')
    print(f'Memoria retornos (GB): {mem_returns:.2f}')
    print(f'Memoria total estimada (GB): {total:.2f}')

estimar_memoria(25_000_000, ret_matrix.shape[1], ret_matrix.shape[0])


In [ ]:
# Monte Carlo (>= 25 millones de agentes) 
start = time.perf_counter()

N_AGENTS = 25_000_000
n_assets = ret_matrix.shape[1]
months = ret_matrix.shape[0]
sum_log = ret_matrix.sum(axis=0).values.astype(np.float32)

rng = np.random.default_rng(42)
weights = rng.random((N_AGENTS, n_assets), dtype=np.float32)
weights = weights / weights.sum(axis=1, keepdims=True)

total_log = weights @ sum_log
cost_log = np.log(1 - 0.0046)
total_log = total_log + months * cost_log

years = months / 12
cagr_agents = np.exp(total_log / years) - 1

strategy_cagr = cagr(strategy_ret)
pct_beaten = (cagr_agents < strategy_cagr).mean()

elapsed = time.perf_counter() - start
print('CAGR estrategia:', strategy_cagr)
print('Porcentaje de agentes aleatorios superados:', pct_beaten)
print('Tiempo de ejecucion (seg):', elapsed)
print('Tiempo de ejecucion (horas):', elapsed / 3600)
print('<= 24h:', (elapsed / 3600) < 24)

plt.figure(figsize=(10, 5))
plt.hist(cagr_agents, bins=100, alpha=0.7, label='Agentes aleatorios')
plt.axvline(strategy_cagr, color='red', linestyle='--', label='Estrategia')
plt.title('Distribucion de CAGR (Monte Carlo)')
plt.legend()
plt.grid(True)
plt.show()
